### Name: Ashish Mulani
### PRN: 260240128007
------------------------------
### Name: Pranita Lad
### PRN: 260240128028

In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, ElasticNet, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import  ColumnTransformer
from sklearn.compose import make_column_selector

from sklearn.impute import SimpleImputer 

import os
os.chdir("D:\CDAC\Machine_Learning\Datasets\Machine_Learning\Datasets") #fetch the dataset from the folder

import warnings
warnings.filterwarnings('ignore')

In [22]:
train = pd.read_csv("train.csv")
train

,id,date,country,store,product,num_sold
0,0,2010-01-01,Canada,Discount Stickers,Holographic Goose,NaN
1,1,2010-01-01,Canada,Discount Stickers,Kaggle,973.0
2,2,2010-01-01,Canada,Discount Stickers,Kaggle Tiers,906.0
3,3,2010-01-01,Canada,Discount Stickers,Kerneler,423.0
4,4,2010-01-01,Canada,Discount Stickers,Kerneler Dark Mode,491.0
...,...,...,...,...,...,...
230125,230125,2016-12-31,Singapore,Premium Sticker Mart,Holographic Goose,466.0
230126,230126,2016-12-31,Singapore,Premium Sticker Mart,Kaggle,2907.0
230127,230127,2016-12-31,Singapore,Premium Sticker Mart,Kaggle Tiers,2299.0
230128,230128,2016-12-31,Singapore,Premium Sticker Mart,Kerneler,1242.0


In [23]:
train.isnull().sum()

id             0
date           0
country        0
store          0
product        0
num_sold    8871
dtype: int64

In [24]:
train = train.dropna(subset=['num_sold'])

In [25]:

X = train.drop(['id', 'date', 'num_sold'], axis=1)
y = train['num_sold']

dum_train = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=26)

In [26]:
ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")

trans = ColumnTransformer(
    transformers=[("OHE", ohe,make_column_selector(dtype_include=object))],remainder="passthrough",
    verbose_feature_names_out=False).set_output(transform="pandas")

In [27]:
X_trn_ohe = trans.fit_transform(X_train)
X_tst_ohe = trans.transform(X_test)

In [28]:
# Linear Regression
lr=LinearRegression()
lr.fit(X_trn_ohe,y_train)
y_pred =lr.predict(X_tst_ohe)
print("R2 =",r2_score(y_test,y_pred))

R2 = 0.7896910334776606


In [29]:
# lasso Regression
alphas=np.linspace(0.001, 5)
scores=[]

for a in alphas:
    lasso = Lasso(alpha = a)
    lasso.fit(X_trn_ohe, y_train)
    y_pred = lasso.predict(X_tst_ohe)
    scores.append([a, r2_score(y_test, y_pred)])
    
df_scores = pd.DataFrame(scores, columns=['alpha','score'])
df_scores.sort_values('score', ascending=False).head()

,alpha,score
0,0.001000,0.789691
1,0.103020,0.789684
2,0.205041,0.789668
3,0.307061,0.789644
4,0.409082,0.789615


In [30]:
# Ridge Regression
alphas=np.linspace(0.001, 5)
scores=[]

for a in alphas:
    ridge = Ridge(alpha = a)
    ridge.fit(X_trn_ohe, y_train)
    y_pred = ridge.predict(X_tst_ohe)
    scores.append([a, r2_score(y_test, y_pred)])
    
df_scores = pd.DataFrame(scores, columns=['alpha','score'])
df_scores.sort_values('score', ascending=False).head()

,alpha,score
0,0.001000,0.789691
1,0.103020,0.789691
2,0.205041,0.789691
3,0.307061,0.789691
4,0.409082,0.789691


In [31]:
alphas=np.linspace(0.001, 5, 20)
ratio=np.linspace(0, 1, 20)
scores=[]

for a in alphas:
    for r in ratio:
        e1 = ElasticNet(alpha = a, l1_ratio = r)
        e1.fit(X_trn_ohe, y_train)
        y_pred = e1.predict(X_tst_ohe)
        scores.append([a, r, r2_score(y_test, y_pred)])
        
df_scores = pd.DataFrame(scores, columns=['alpha', 'score', 'l1_ratio'])
df_scores.sort_values('score', ascending=False).head()

,alpha,score,l1_ratio
399,5.000000,1.0,0.780680
199,2.368947,1.0,0.787650
59,0.527211,1.0,0.789574
79,0.790316,1.0,0.789448
99,1.053421,1.0,0.789272


# Inferencing

In [32]:
tst = pd.read_csv("test.csv")
tst

,id,date,country,store,product
0,230130,2017-01-01,Canada,Discount Stickers,Holographic Goose
1,230131,2017-01-01,Canada,Discount Stickers,Kaggle
2,230132,2017-01-01,Canada,Discount Stickers,Kaggle Tiers
3,230133,2017-01-01,Canada,Discount Stickers,Kerneler
4,230134,2017-01-01,Canada,Discount Stickers,Kerneler Dark Mode
...,...,...,...,...,...
98545,328675,2019-12-31,Singapore,Premium Sticker Mart,Holographic Goose
98546,328676,2019-12-31,Singapore,Premium Sticker Mart,Kaggle
98547,328677,2019-12-31,Singapore,Premium Sticker Mart,Kaggle Tiers
98548,328678,2019-12-31,Singapore,Premium Sticker Mart,Kerneler


In [36]:
X_ohe=trans.fit_transform(X)
bm = Ridge(alpha= 0.001000)
bm.fit(X_ohe, y)

Ridge(alpha=0.001)

In [38]:
test_ohe=trans.transform(tst)
y_pred=bm.predict(test_ohe)
y_pred

array([-244.65274366,  917.18127107,  703.14079859, ..., 1429.00780424,
        966.0064382 , 1061.68093936])

In [40]:
ss = pd.read_csv("sample_submission.csv")
ss['num_sold']=y_pred
ss.to_csv("sbt_log_18_4.csv", index=False)
ss

,id,num_sold
0,230130,-244.652744
1,230131,917.181271
2,230132,703.140799
3,230133,240.139433
4,230134,335.813934
...,...,...
98545,328675,481.214262
98546,328676,1643.048277
98547,328677,1429.007804
98548,328678,966.006438
